# 3D toric code — L=4 magnetic (h_x) transition line across the XZ plane

Locates the **topological → x-polarized (m-condensed / trivial)** transition driven by
$h_x$ at each fixed $h_z$, from the **broad L=4 hx-sweeps** (`phase_hz{0.0…1.1}/L4`). This
is the (expected **1st-order**) *magnetic* line — the $e\!\leftrightarrow\!m$ mirror of the
continuous $h_z$ (electric) transition. The 3D TC is **not self-dual**, so $h_x^c(h_z)$ is
its own curve.

Everything here is **free** (no GPU): each run JSON already carries the movers for an
hx-sweep — $\langle M_x\rangle$ (rises $0\to1$), $\langle B_p\rangle$ (drops, topological
order), plus `Vscore` (criticality proxy) and $E_0$ (energy-kink). Pulled to
`results/xz_line_L4/energy_L4_hz*.json` by `check_convergence.py --dump` on a login node.

**Order parameter caveat:** at L=4 the curves are finite-size *rounded*, so this notebook
locates $h_x^c(h_z)$ and defines educated windows — it does **not** by itself prove the
transition is 1st-order (that needs the hysteresis ramps / energy-kink FSS).

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
DIR  = f"{ROOT}/xz_line_L4"        # energy_L4_hz*.json dumps (hx, E, mx, A_v, B_p, Vscore)
L    = 4

# --- locator knob: half-width (in hx) of the recommended larger-L window around h_c ---
PAD  = 0.25                         # window = consensus h_c ± PAD
NPTS = 7                            # points per larger-L window (for the printed grid)

# --- FINALIZE HERE: after eyeballing §3–§5, override any per-hz window you want. ---
#     Leave a key out to accept the auto (consensus h_c ± PAD) window.
WINDOWS_FINAL = {
    # 0.0: (0.45, 0.95),
    # 0.5: (0.55, 1.05),
    # 1.1: (1.30, 1.80),
}

# drop specific (hz, hx) points from the locators if a run looks off
EXCLUDE = {}   # {hz: [hx, ...]}

## 2 · Data library

One record per $h_z$ cut into `DATA[hz]` with sorted `hx`, `mx`, `bp`, `vs`, `E` arrays.

In [ ]:
# ====================== 2 · DATA LIBRARY ======================
DATA = {}
for jp in sorted(glob.glob(os.path.join(DIR, "energy_L4_hz*.json"))):
    d = json.load(open(jp))
    hz = float(os.path.basename(jp).split("hz")[1].rsplit(".json", 1)[0])
    hx = np.array(d["field"], float)
    o = np.argsort(hx)
    keep = ~np.isin(np.round(hx[o], 6), np.round(EXCLUDE.get(hz, []), 6))
    DATA[hz] = dict(
        hx=hx[o][keep],
        mx=np.array(d["mx"], float)[o][keep],
        bp=np.array(d["B_p"], float)[o][keep],
        vs=np.array(d["Vscore"], float)[o][keep],
        E=np.array(d["E"], float)[o][keep])
if not DATA:
    raise SystemExit(f"no energy_L4_hz*.json in {DIR}")
HZS = sorted(DATA)
print(f"loaded {len(HZS)} hz cuts: {HZS}")
for hz in HZS:
    hx = DATA[hz]["hx"]
    print(f"  hz={hz:<4}  {len(hx):2d} pts   hx ∈ [{hx.min():.3g}, {hx.max():.3g}]")

## 3 · Order parameters vs $h_x$ (colored by $h_z$)

$\langle M_x\rangle$ rises through the transition; $\langle B_p\rangle$ (plaquette stabilizer)
collapses; `Vscore` peaks near the pseudo-critical field. As $h_z$ grows the whole family
shifts to the **right** — $h_x^c$ increases with $h_z$.

In [ ]:
# ====================== 3 · CURVES ======================
cmap = plt.cm.viridis; norm = plt.Normalize(min(HZS), max(HZS))
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
for hz in HZS:
    c = cmap(norm(hz)); D = DATA[hz]
    ax[0,0].plot(D["hx"], D["mx"], "-o", ms=3, color=c, label=f"hz={hz}")
    ax[0,1].plot(D["hx"], D["bp"], "-o", ms=3, color=c)
    ax[1,0].plot(D["hx"], D["vs"], "-o", ms=3, color=c)
    hm = 0.5*(D["hx"][1:] + D["hx"][:-1]); dmx = np.diff(D["mx"])/np.diff(D["hx"])
    ax[1,1].plot(hm, dmx, "-o", ms=3, color=c)
ax[0,0].axhline(0.5, ls=":", c="grey", lw=0.8)
ax[0,0].set(xlabel="$h_x$", ylabel=r"$\langle M_x\rangle$", title="Magnetization (order param)")
ax[0,0].legend(fontsize=7, ncol=2)
ax[0,1].set(xlabel="$h_x$", ylabel=r"$\langle B_p\rangle$", title="Plaquette stabilizer (topological order)")
ax[1,0].set(xlabel="$h_x$", ylabel="Vscore", title="Vscore (criticality proxy)")
ax[1,1].set(xlabel="$h_x$", ylabel=r"$dM_x/dh_x$", title="Magnetic susceptibility proxy")
plt.tight_layout(); plt.show()

## 4 · Transition locators

Three model-free estimates of $h_x^c$ per $h_z$, each parabola-refined at its extremum:
* **susc** — $\arg\max\; dM_x/dh_x$ (magnetic susceptibility peak)
* **vscore** — $\arg\max\;\mathrm{Vscore}$ (empirical criticality proxy)
* **mid** — $\langle M_x\rangle = 0.5$ crossing (linear interp)

Their mean is the **consensus** $h_x^c$; their spread is the L=4 locator uncertainty. A blank
estimator means its extremum/crossing fell at a window edge (not bracketed) — a signal the
sweep window sat slightly too high/low at that $h_z$.

In [ ]:
# ====================== 4 · LOCATORS ======================
def parab_peak(x, y):
    """x* of the parabola through the max and its neighbours; (nan, False) if edge."""
    i = int(np.argmax(y))
    if i == 0 or i == len(y)-1:
        return np.nan, False
    x0,x1,x2 = x[i-1:i+2]; y0,y1,y2 = y[i-1:i+2]
    d = (x0-x1)*(x0-x2)*(x1-x2)
    a = (x2*(y1-y0) + x1*(y0-y2) + x0*(y2-y1)) / d
    b = (x2*x2*(y0-y1) + x1*x1*(y2-y0) + x0*x0*(y1-y2)) / d
    return (x1 if a == 0 else -b/(2*a)), True

def mid_cross(x, m, lvl=0.5):
    for i in range(len(m)-1):
        if (m[i]-lvl)*(m[i+1]-lvl) <= 0 and m[i+1] != m[i]:
            t = (lvl-m[i])/(m[i+1]-m[i]); return x[i]+t*(x[i+1]-x[i])
    return np.nan

LOC = {}
print(f"{'hz':>5} {'window':>14} {'susc':>7} {'vscore':>7} {'mid':>7} {'h_c':>7} {'±sd':>6}")
for hz in HZS:
    D = DATA[hz]; hx = D["hx"]
    dmx = np.gradient(D["mx"], hx)
    hc_s, brs = parab_peak(hx, dmx)
    hc_v, brv = parab_peak(hx, D["vs"])
    hc_m = mid_cross(hx, D["mx"])
    ests = [v for v in (hc_s if brs else np.nan, hc_v if brv else np.nan, hc_m) if np.isfinite(v)]
    hc = float(np.mean(ests)) if ests else np.nan
    sd = float(np.std(ests)) if len(ests) > 1 else np.nan
    LOC[hz] = dict(susc=hc_s if brs else np.nan, vscore=hc_v if brv else np.nan,
                   mid=hc_m, hc=hc, sd=sd)
    f = lambda v: f"{v:7.3f}" if np.isfinite(v) else "      -"
    print(f"{hz:>5} {f'[{hx.min():.3g},{hx.max():.3g}]':>14} "
          f"{f(LOC[hz]['susc'])} {f(LOC[hz]['vscore'])} {f(hc_m)} {f(hc)} {f(sd)}")

## 5 · The L=4 transition line in the XZ plane

Consensus $h_x^c$ (with locator spread as the horizontal error bar) vs $h_z$; the three
individual estimators overplotted. Flat at $h_x^c\approx0.67$ for $h_z\lesssim0.4$, then
bending up steeply.

In [ ]:
# ====================== 5 · PHASE BOUNDARY ======================
fig, axb = plt.subplots(figsize=(6.5, 6))
hc = np.array([LOC[hz]["hc"] for hz in HZS])
sd = np.array([LOC[hz]["sd"] for hz in HZS])
axb.errorbar(hc, HZS, xerr=np.nan_to_num(sd), fmt="s-", capsize=3, color="crimson", zorder=2)
for hz in HZS:
    axb.scatter([LOC[hz]["susc"], LOC[hz]["vscore"], LOC[hz]["mid"]], [hz]*3,
                s=16, c=["tab:blue", "tab:green", "tab:orange"], zorder=3)
axb.scatter([], [], c="tab:blue", label="susc (dM_x/dh_x)")
axb.scatter([], [], c="tab:green", label="vscore peak")
axb.scatter([], [], c="tab:orange", label="M_x=0.5")
axb.annotate("TOPOLOGICAL\n(deconfined)", (min(hc)-0.02, 0.85), ha="right", color="navy", fontsize=9)
axb.annotate("TRIVIAL\n(x-polarized)", (max(hc)+0.02, 0.25), color="darkred", fontsize=9)
axb.set(xlabel=r"$h_x^c$", ylabel=r"$h_z$", title=f"L={L} magnetic transition line (XZ plane)")
axb.grid(alpha=0.3); axb.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

## 6 · Finalize the L=5/6/7 windows

Auto window = consensus $h_x^c \pm$ `PAD`; override any cut in `WINDOWS_FINAL` (§1). This cell
prints the per-$h_z$ hx grids and the `HZ`/`HXLO`/`HXHI`/`HXN` env-var lines to paste into
`nersc/submit_nqs_hx_sweep.sh`. Re-run §1 after editing `WINDOWS_FINAL`.

In [ ]:
# ====================== 6 · WINDOW FINALIZER ======================
def window(hz):
    if hz in WINDOWS_FINAL:
        return WINDOWS_FINAL[hz]
    hc = LOC[hz]["hc"]
    return (round(hc-PAD, 3), round(hc+PAD, 3)) if np.isfinite(hc) else (np.nan, np.nan)

print(f"{'hz':>5} {'h_c':>7} {'window':>16} {'grid (hx points)':>40}")
for hz in HZS:
    lo, hi = window(hz)
    grid = np.round(np.linspace(lo, hi, NPTS), 3) if np.isfinite(lo) else []
    src = "manual" if hz in WINDOWS_FINAL else "auto"
    print(f"{hz:>5} {LOC[hz]['hc']:>7.3f} {f'[{lo}, {hi}] ({src})':>16} "
          f"{'  '.join(f'{g:g}' for g in grid):>40}")

print("\n# --- submit env lines (one hx-sweep array per hz) ---")
for hz in HZS:
    lo, hi = window(hz)
    if not np.isfinite(lo):
        continue
    print(f"HZ={hz} HXLO={lo} HXHI={hi} HXN={NPTS} L=<5|6|7> bash nersc/submit_nqs_hx_sweep.sh")